[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/controlled_noisy.ipynb)

# MISDA — noisy controlled benchmark

This notebook is a presentation front end for `misda.benchmarks.validation.analyze_controlled_noisy_problem`. It repeats the canonical 13-case controlled benchmark at the fixed scale-relative observation-noise condition `sigma=0.10`. Ground truth is always derived from clean `Z`; MISDA sees only noisy `Y`.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
target = f"{repo_root}[benchmarks]" if repo_root is not None else "misda[benchmarks] @ git+https://github.com/monacofj/misda.git@main"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import misda
from misda.benchmarks.validation import analyze_controlled_noisy_problem

N = 300
SEED = 123
OBSERVATION_SEED = 456
SIGMA = 0.10


## Scale-relative observation noise

For each clean objective `Z_j`, observation follows `Y_j = Z_j + sigma * std(Z_j) * epsilon_j`. The sample and observation streams use distinct fixed seeds. `sigma=0.10` is a reproducible reference condition, not a robustness threshold.


In [ ]:
noisy_results = {}

def run_case(problem_id):
    item = analyze_controlled_noisy_problem(
        problem_id, n=N, seed=SEED, observation_seed=OBSERVATION_SEED, sigma=SIGMA
    )
    print(item["benchmark_obj"].report())
    item["ranking_obj"].mis().graph_plot()
    noisy_results[problem_id] = item
    return item["benchmark_obj"]


## Case 1 - Independent objectives

**Noise focus.** Twenty independent objectives; noise tests false redundancy among unrelated columns.


In [ ]:
case_1 = run_case("independence")


## Case 2 - Complete positive redundancy

**Noise focus.** One common factor copied across objectives; noise tests whether the one-dimensional signal remains grouped.


In [ ]:
case_2 = run_case("total_redundancy")


## Case 3 - Four redundant blocks

**Noise focus.** Four independent 5-objective blocks; noise tests within-block cohesion and between-block separation.


In [ ]:
case_3 = run_case("blocks_4x5")


## Case 4 - Two redundant blocks

**Noise focus.** Two large redundant blocks; noise tests whether the two groups remain distinguishable.


In [ ]:
case_4 = run_case("blocks_2x10")


## Case 5 - Mixed independent and redundant objectives

**Noise focus.** Independent objectives coexist with two redundant blocks; noise tests both isolation and grouping.


In [ ]:
case_5 = run_case("mixed_independent_and_blocks")


## Case 6 - Nonlinear monotonic redundancy

**Noise focus.** One latent factor generates nonlinear monotonic transforms; noise tests stability of the one-dimensional structure.


In [ ]:
case_6 = run_case("monotonic_redundancy")


## Case 7 - Antagonistic linear groups

**Noise focus.** Two positively redundant groups oppose each other; noise tests signed conflict and within-group redundancy.


In [ ]:
case_7 = run_case("antagonistic_linear_groups")


## Case 8 - Trade-off with redundant families

**Noise focus.** Two generating variables create several observable families; noise tests recovery without overcounting families.


In [ ]:
case_8 = run_case("tradeoff_redundancies")


## Case 9 - Nonlinear redundant blocks

**Noise focus.** Four nonlinear redundant blocks; noise compounds nonlinear variation with observation error.


In [ ]:
case_9 = run_case("nonlinear_blocks_4x5")


## Case 10 - Antagonistic nonlinear groups

**Noise focus.** Nonlinear redundancy and antagonism occur together; noise tests both simultaneously.


In [ ]:
case_10 = run_case("antagonistic_nonlinear_groups")


## Case 11 - Overlapping latent factors

**Noise focus.** Observable families overlap in two latent factors; noise tests whether the two-dimensional organization remains visible.


In [ ]:
case_11 = run_case("overlapping_factors")


## Case 12 - Transitive positive chain

**Noise focus.** Known adversarial chain; the key question is whether TRANSITIVE_CHAINING remains signaled under noise.


In [ ]:
case_12 = run_case("transitive_chain")


## Case 13 - Regime-switching dependence

**Noise focus.** Known adversarial regime mixture; the key question is whether HIDDEN_SPECTRAL_STRUCTURE remains signaled under noise.


In [ ]:
case_13 = run_case("regime_switching")


# Suite summary

The table below gives the cross-case summary for the fixed noisy reference condition.


In [ ]:
noisy_summary = misda.compile_benchmark_summary(noisy_results)
noisy_summary
